# 📘 부록: OKF 비서 만들기 2. 코딩 에이전트와 스킬로 위키 키우기

부록 02의 비서는 위키를 읽고 답만 했습니다. 이번에는 위키를 키웁니다. 새 자료가 오면 위키에 넣고, 위키를 점검하는 일을 Claude Code와 Codex가 스킬로 합니다. 스킬을 구성하고, 두 도구가 찾는 폴더에 등록한 뒤, 터미널에서 지시해 활용합니다. 두 코딩 에이전트는 등록된 `SKILL.md`를 스스로 찾아 읽고, 위키 파일을 직접 고치고, 검사 스크립트까지 실행합니다.

<img src="images/14_agent_skills_ingest.png" width="1050" alt="사람이 새 자료를 넣으라고 지시하면 에이전트가 스킬 okf-ingest를 고르고, raw/의 원본은 읽기만 한 뒤 위키의 문서, 목차, 변경 기록을 고치고 check.py로 검사합니다. Claude Code는 .claude/skills/, Codex는 .agents/skills/에서 같은 SKILL.md를 찾습니다.">

**공식 문서:** [Claude Code 스킬](https://code.claude.com/docs/en/skills), [Codex 스킬](https://developers.openai.com/codex/skills)

**학습 목표**

1. **스킬 폴더(`SKILL.md`와 스크립트)를 구성합니다.**
2. **같은 스킬을 Claude Code와 Codex가 찾는 폴더에 등록합니다.**
3. **등록한 스킬로 새 자료를 위키에 넣고 점검하게 한 뒤, 바뀐 줄과 검사 결과를 확인합니다.**

## 1. 스킬 구성

### 1-1. 스킬과 점진적 공개

**스킬(skill)** 은 에이전트가 특정 일을 할 때 펼쳐 보는 업무 설명서입니다. 폴더 하나에 `SKILL.md`와, 그 절차가 쓰는 스크립트나 자료를 담습니다. `SKILL.md`는 맨 위 frontmatter(`name`, `description`)와 본문(절차, 규칙)으로 이루어집니다. 이 형식(Agent Skills)은 Claude Code, Codex 같은 코딩 에이전트가 그대로 읽습니다.

<img src="images/15_agent_skills_disclosure.png" width="1050" alt="코딩 에이전트는 처음에 스킬의 이름과 설명만 알고, 일을 맡으면 SKILL.md 본문을 펼치며, 본문이 가리키는 스크립트 check.py는 실행할 때만 씁니다.">

- **1단계, 처음부터 아는 것:** 스킬마다 이름과 한 줄 설명만 압니다.
- **2단계, 필요할 때 펼침:** 지시가 설명과 맞으면 그 스킬의 `SKILL.md` 본문을 읽습니다.
- **3단계, 지침이 가리키는 것:** 본문이 가리키는 스크립트는 실행할 때만 씁니다. 스크립트 코드를 읽지 않아도 실행 결과만으로 쓸 수 있습니다.

스킬이 열 개, 스무 개로 늘어도 처음 맥락에는 설명 한 줄씩만 들어갑니다. 도구가 동작 하나라면, 스킬은 그 도구들을 어떤 순서와 규칙으로 쓸지 적은 지침입니다. LLM 위키의 세 층 가운데 스키마(위키 규칙 문서)에 해당합니다.

### 1-2. 스킬 폴더의 구성

스킬 하나는 폴더 하나입니다. 폴더 안에 `SKILL.md`와, 그 절차가 쓰는 스크립트를 둡니다. 이 부록의 스킬 두 개는 작업 폴더 `agent_wiki/`의 스킬 폴더에 있습니다(2절).

~~~text
agent_wiki/.claude/skills/
├── okf-ingest/
│   ├── SKILL.md            새 자료 하나를 위키에 넣는 절차와 쓰기 규칙
│   └── scripts/check.py    검사 스크립트
└── okf-lint/
    ├── SKILL.md            위키를 점검해 보고하는 절차(문서는 고치지 않음)
    └── scripts/check.py    같은 검사 스크립트
~~~

- **frontmatter:** `name`(스킬 이름), `description`(언제 쓰는 스킬인지), `allowed-tools`(Claude Code가 묻지 않고 쓸 도구)입니다. 1단계에서 에이전트가 아는 것이 이 칸입니다.
- **본문:** 순서와 규칙입니다. 에이전트가 그대로 따릅니다.
- **스크립트:** 본문이 실행하라고 한 파일입니다. 에이전트는 코드를 읽지 않아도 실행 결과만으로 씁니다.

### 1-3. 넣기 스킬

넣기 스킬은 번들과 고칠 문서를 고르고, 문서와 frontmatter, 폴더 목차, 변경 기록을 직접 쓴 뒤 검사합니다. 어느 번들에도 맞지 않는 자료는 새 번들을 만듭니다. 쓰기 규칙은 모델이 실제로 틀렸던 것(자료에 없는 날짜를 붙임, 이력을 본문에 씀)을 막도록 적었습니다. 자료 한 건마다 아래 순서를 따릅니다.



에이전트가 일을 맡으면 펼쳐 읽는 넣기 스킬의 본문입니다. 위 순서와 규칙이 모두 여기에 있습니다.

`agent_wiki/.claude/skills/okf-ingest/SKILL.md`

~~~markdown
---
name: okf-ingest
description: raw/의 새 자료 하나를 OKF 위키(bundles/)에 넣는다. 사람이 새 자료를 위키에 넣으라고 할 때 쓴다.
allowed-tools: Read, Edit, Write, Glob, Bash(python3 *)
---
# OKF 위키에 새 자료 넣기

작업 폴더의 AGENTS.md 규칙을 따른다. 오늘 날짜는 2026-11-02다. 스크립트는 이 SKILL.md 옆 `scripts/check.py`다. 작업 폴더에서 Claude Code는 `python3 .claude/skills/okf-ingest/scripts/check.py`, Codex는 `python3 .agents/skills/okf-ingest/scripts/check.py`로 실행한다.

## 순서

1. 자료(`raw/<파일 이름>`)를 읽는다. 자료는 고치지 않는다.
2. 번들마다 맨 위 `index.md`를 읽고, 자료의 주제를 그 목차의 폴더 설명과 견줘 자료를 반영할 번들을 고른다. 문서 안에 같은 이름(게임, 사람, 프로젝트 이름)이 나온다는 이유만으로 그 번들이나 문서를 고르지 않는다. 광고, 스팸처럼 남길 필요가 없는 자료면 "건너뜀: 이유"라고 답하고 끝낸다. 남길 자료인데 어느 번들의 폴더 설명에도 맞지 않으면 아래 '새 번들 만들기'대로 새 번들을 만들고 7번으로 간다(새 번들 만들기가 목차와 변경 기록까지 쓴다).
3. 고른 번들에서 폴더 `index.md`를 읽고, 자료가 바꾸거나 더하는 사실이 들어갈 문서를 고른다. 회의 메모라면 회의 정리 문서가 아니라 그 결정으로 내용이 바뀌는 문서를 고른다. 자료가 다루는 대상의 문서가 없으면 그 폴더에 새 문서를 만든다.
4. 고치는 문서면 원래 문서를 읽고, 아래 쓰기 규칙과 frontmatter 규칙대로 문서를 쓴다.
5. 새 문서면 그 폴더 `index.md`에 `* [제목](파일.md) - 한 줄 설명` 한 줄을 더한다.
6. 번들 `log.md`의 `## 2026-11-02` 제목 아래(없으면 맨 위 제목 바로 아래에 만든다)에 한 줄을 적는다. 고친 문서는 `* **Update**: <자료 파일 이름>를 반영해 [제목](/폴더/파일.md) 문서를 고쳤습니다.`, 새 문서는 `* **Creation**: <자료 파일 이름>를 반영해 [제목](/폴더/파일.md) 문서를 만들었습니다.`
7. check.py로 검사하고, 문제가 나오면 고친 뒤 다시 검사한다.
8. 한국어로 한 줄 답한다: 작업(Update, Creation, 건너뜀), 번들/개념 ID, 상태. 새 번들을 만들었으면 "새 번들"이라고 밝힌다.

## 쓰기 규칙

- 고치는 문서면 원래 본문의 구조와 내용을 유지하고, 자료가 바꾼 사실만 고치거나 더한다. title은 그대로 두고, description은 자료 때문에 틀리게 된 사실만 고친다. description은 한 줄 설명이라 각주 표시를 붙이지 않는다.
- 새 문서면 같은 폴더 문서들의 양식(제목 단계, 목록 모양)을 따른다.
- 다른 문서는 `[이름](/폴더/파일.md)` 꼴로 링크한다. 링크는 번들 맨 위 기준이라 번들 이름은 넣지 않는다(`/people/jiho.md`는 맞고 `/studio/people/jiho.md`는 끊긴다). 태그는 영문 소문자로 쓴다.
- 자료에서 온 사실 뒤에는 각주 표시 `[^자료 파일 이름에서 확장자를 뺀 것]`을 붙이고, 바뀐 사실에 붙어 있던 옛 각주 표시는 지운다. 본문 맨 끝에 각주 정의 줄 `[^각주 이름]: 자료 파일 이름`을 둔다.
- 바뀐 값은 새 값만 적는다(예: '정식 출시: 12월 4일'). '무엇에서 무엇으로 바뀜' 같은 이력은 본문과 description에 쓰지 않는다. 이력은 log.md에 남는다.
- 자료에 없는 연도, 날짜, 숫자를 붙이지 않는다. 바꾸지 않는 문장은 글자 그대로 둔다. 자료에 없는 사실은 지어내지 않는다. 한국어로 쓴다.

## 새 번들 만들기

어느 번들의 폴더 설명에도 맞지 않는 자료만 새 번들로 만든다. 기존 번들의 폴더 설명에 맞으면 그 번들에 넣는다.

- 폴더: `bundles/<영문 소문자 이름>/`(예: `support`).
- 맨 위 `index.md`: frontmatter는 `okf_version: "0.2"` 하나만 두고, 본문은 `# 위키 제목`과 폴더마다 `* [폴더 이름](폴더/) - 한 줄 설명`이다.
- 폴더: 자료의 대상에 맞는 폴더 하나를 만들고, 그 안에 폴더 `index.md`(`# 폴더 이름`과 문서 목록, frontmatter 없음)와 새 문서를 둔다. 새 문서의 `type`은 대상에 맞게 정한다.
- `log.md`: `# 위키 제목 변경 기록`, `## 2026-11-02`, 그 아래 `* **Initialization**: 번들을 만들었습니다.`와 Creation 한 줄.

## frontmatter 규칙

- 고치는 문서: 원래 키는 살린다. `verified`는 지운다(내용이 바뀌었으니 사람이 다시 확인한다). `generated`는 `{ by: "도구 이름", at: "2026-11-02T09:00:00+09:00" }`로 바꾼다(도구 이름은 claude-code 또는 codex). `sources`에 `{ id: 각주 이름, resource: "raw://자료 파일 이름", title: 자료 파일 이름 }` 항목을 더한다. 자료 때문에 대상이 더는 현재가 아니면(종료, 폐기, 퇴사, 해지) `status: deprecated`로 바꾼다.
- 새 문서: `type`은 같은 폴더 문서의 type을 따르고, `title`, `description`, `tags`, `status: draft`, `generated`, `sources`를 적는다. 파일 이름은 영문 소문자와 밑줄로 짓는다.
~~~

### 1-4. 점검 스킬과 검사 스크립트

- **점검 스킬(`okf-lint`):** `check.py`를 실행하고, 오늘 바뀐 문서와 링크로 이어진 문서를 읽어 서로 다른 값을 찾아 표로 보고합니다. 문서는 고치지 않습니다.

<img src="images/17_agent_lint_steps.png" width="1050" alt="점검 스킬 okf-lint의 네 단계: 1 검사 실행(check.py로 적합성, 끊긴 링크, 만료, 없는 원문), 2 오늘 바뀐 문서(log.md의 오늘 날짜), 3 이웃 문서와 대조(링크로 이어진 문서, 회의 정리 문서는 빼고), 4 표로 보고(검사 결과, 모순). 문서는 고치지 않고 사람이 정하게 알립니다.">

- **검사 스크립트(`check.py`):** 적합성(부록 01 6절), 끊긴 링크, 만료, 없는 원문(`raw://`)을 찾습니다. 표준 라이브러리만 써서 어느 파이썬으로도 돕니다.

## 2. 스킬 등록

### 2-1. 도구마다 스킬을 찾는 폴더

스킬을 등록한다는 것은 도구가 찾는 폴더에 스킬 폴더를 두는 것입니다. 두 도구는 찾는 폴더가 다릅니다.

| | Claude Code | Codex |
|---|---|---|
| 스킬 폴더 | 작업 폴더의 `.claude/skills/<이름>/SKILL.md` | 작업 폴더의 `.agents/skills/<이름>/SKILL.md` |
| frontmatter | 모든 키가 선택이고 `description`을 권장합니다 | `name`, `description`이 꼭 있어야 합니다 |
| 직접 부르기 | `/okf-ingest` | `$okf-ingest` |
| 파일 고치기 | 스킬의 `allowed-tools`에 적은 도구는 묻지 않고 쓰고, 그 밖의 도구는 쓸 때 묻습니다 | 처음 실행할 때 이 폴더에서 작업해도 되는지 묻습니다. 허용하면 작업 폴더 안의 파일을 고칩니다 |

그래서 같은 스킬을 두 폴더에 모두 넣습니다. Claude Code만 쓰는 키(`allowed-tools`)는 Codex가 모르는 키라 무시합니다.

### 2-2. 이 부록의 작업 폴더

작업 폴더 `agent_wiki/`는 부록 폴더에 미리 만들어 두었습니다. 규칙 파일, 번들, 원본 자료가 들어 있고, 스킬 두 개가 두 도구의 스킬 폴더에 등록되어 있습니다. 원본 `data/`는 그대로 두고 이 폴더의 위키를 고칩니다.

~~~text
agent_wiki/
├── AGENTS.md                 규칙 파일(부록 02)
├── bundles/                  위키(작업 사본). 에이전트가 고치는 곳
├── raw/                      원본 자료. 읽기만 함
├── .claude/skills/           Claude Code가 찾는 스킬 폴더
│   ├── okf-ingest/
│   └── okf-lint/
└── .agents/skills/           Codex가 찾는 스킬 폴더
    ├── okf-ingest/
    └── okf-lint/
~~~

등록이 됐는지는 도구의 대화 창에서 확인합니다. `agent_wiki` 폴더에서 `claude`를 실행해 `/`를 치면 명령 목록에 `/okf-ingest`, `/okf-lint`가 보이고, `codex`를 실행해 `/skills`를 치면 두 스킬이 보입니다. 실습을 처음부터 다시 하려면 부록 폴더를 다시 받아 `agent_wiki`를 처음 상태로 되돌립니다.

## 3. 스킬 활용

### 3-1. 새 자료 한 건 넣기

**준비:** Claude Code나 Codex CLI가 설치되어 로그인되어 있어야 합니다(부록 02 1-2). `agent_wiki` 폴더에서 터미널을 열고 `claude`(Claude Code) 또는 `codex`(Codex)를 실행합니다. 폴더를 믿을지, 파일을 고쳐도 될지 묻는 창이 나오면 허용합니다. 스킬이 `python3`로 검사 스크립트를 실행하므로, Windows에서 `python3`가 없으면 `SKILL.md`의 `python3`(frontmatter와 본문)를 `python`으로 바꿔 씁니다.

대화 창에 지시하는 방법은 두 가지입니다. 아래 **⌨️ 입력**은 대화 창에 치는 지시이고, **💬 답**은 이 부록을 만들 때 실제로 나온 답입니다. 문장은 도구와 실행마다 조금씩 다릅니다.

| 지시 방법 | Claude Code 대화 창 | Codex 대화 창 |
|---|---|---|
| 스킬 직접 부르기 | `/okf-ingest raw/01_meeting_2026-10-31.md` | `$okf-ingest raw/01_meeting_2026-10-31.md` |
| 지시어로 맡기기 | `raw/01_meeting_2026-10-31.md 자료를 OKF 위키에 넣어 줘` | `raw/01_meeting_2026-10-31.md 자료를 OKF 위키에 넣어 줘` |

지시어로 맡기면 에이전트가 스킬의 `description`("새 자료 하나를 OKF 위키에 넣는다")과 지시를 견줘 `okf-ingest`를 스스로 고릅니다. 직접 부르면 고르는 단계 없이 바로 그 스킬을 씁니다. Claude Code는 지시어로 맡기면 스킬의 `allowed-tools`가 적용되지 않아, 파일을 고치거나 검사 스크립트를 실행할 때 허락을 묻습니다. 허용하면 이어서 진행합니다.

⌨️ 입력 (Claude Code, 스킬 직접 부르기)

~~~text
/okf-ingest raw/01_meeting_2026-10-31.md
~~~

💬 답 (Claude Code)

~~~text
Update: studio/projects/chorong_launch 문서에 회의 내용을 넣었습니다. 바뀐 일정은 스토어 심사 제출 11월 13일, 정식 출시 12월 4일입니다. 박서연의 환불 오류 수정(11월 10일까지)도 남은 일에 더했습니다. 상태는 stable로 두었고 verified는 지웠으니 사람이 다시 확인해야 합니다. log.md에도 한 줄 적었습니다. check.py를 돌리니 끊긴 링크와 frontmatter 문제는 없었고, 원래 있던 playbooks/server_incident의 만료(stale_after 2026-10-01) 1건만 나왔습니다. 이 건은 이번 작업과 관계없어 그대로 두었습니다.
~~~

⌨️ 입력 (Codex, 지시어로 맡기기)

~~~text
raw/01_meeting_2026-10-31.md 자료를 위키에 넣어 줘
~~~

💬 답 (Codex)

~~~text
Update: `studio/projects/chorong_launch`, `studio/playbooks/launch_checklist` (stable)에 일정과 환불 오류 수정 담당·기한을 반영했습니다. 검사에서 수정 문서는 문제가 없었고, 기존 `studio/playbooks/server_incident.md`의 만료 표시 1건이 보고됐습니다.
~~~

이 부록을 만들 때 Claude Code가 바꾼 줄은 아래와 같습니다(`-`는 지운 줄, `+`는 더한 줄). 직접 해 본 뒤에는 `agent_wiki/bundles/studio/projects/chorong_launch.md`와 `log.md`를 열어 같은 곳이 바뀌었는지 봅니다.

~~~diff
===== studio/projects/chorong_launch.md
-generated: { by: "starpixel-secretary/0.1", at: "2026-10-27T18:00:00+09:00" }
-verified: { by: "human:yuna", at: "2026-10-28T09:30:00+09:00" }
+  - { id: 01_meeting_2026-10-31, resource: "raw://01_meeting_2026-10-31.md", title: 01_meeting_2026-10-31.md }
+generated: { by: "claude-code", at: "2026-11-02T09:00:00+09:00" }
-* 스토어 심사 제출: 11월 6일[^weekly-1027]
-* 정식 출시: 11월 20일[^weekly-1027]
+* 스토어 심사 제출: 11월 13일[^01_meeting_2026-10-31]
+* 정식 출시: 12월 4일[^01_meeting_2026-10-31]
-[^weekly-1027]: 10월 27일 주간 회의
+* 결제 연동 테스트에서 나온 환불 처리 오류 수정: [박서연](/people/seoyeon.md), 11월 10일까지[^01_meeting_2026-10-31]
+
+[^01_meeting_2026-10-31]: 01_meeting_2026-10-31.md

===== studio/log.md
+
+## 2026-11-02
+* **Update**: 01_meeting_2026-10-31.md를 반영해 [초롱 원정대 출시](/projects/chorong_launch.md) 문서를 고쳤습니다.
+* **Creation**: 02_mail_new_member.txt를 반영해 [서은채](/people/eunchae.md) 문서를 만들었습니다.
~~~

**결과 읽기**
- **고친 문서:** 프로젝트 문서(`projects/chorong_launch`)의 심사 제출일과 정식 출시일이 바뀌고, 각주가 새 자료를 가리킵니다.
- **frontmatter:** `verified`가 지워지고(내용이 바뀌어 사람이 다시 확인해야 함), `generated`가 도구 이름으로, `sources`에 `raw://` 출처가 더해집니다.
- **변경 기록:** 번들 `log.md`의 `## 2026-11-02` 아래에 한 줄이 생깁니다.
- **도구마다 다른 점:** 이 부록을 만들 때 Codex는 프로젝트 문서와 함께 출시 체크리스트의 출시일까지 고쳤고, 프로젝트 문서의 옛 출처(`weekly-1027`)를 지우고 새 출처로 바꿨습니다. 스킬은 원래 키를 살리고 출처를 더하라고 했으니 규칙을 어긴 것입니다. 에이전트가 직접 쓰면 이런 차이가 생기므로, 바뀐 줄을 사람이 확인합니다.

### 3-2. 나머지 네 건 넣기

자료마다 위키에 들어가는 갈래가 다릅니다. 새 팀원 메일과 ARPPU 메모는 새 문서, 결제 전환율 중단 메모는 기존 문서를 고쳐 `deprecated`, 광고 메일은 건너뜁니다.

⌨️ 입력 (Claude Code). 한 줄씩 차례로 칩니다.

~~~text
/okf-ingest raw/02_mail_new_member.txt
/okf-ingest raw/03_data_memo_arppu.md
/okf-ingest raw/04_data_memo_conversion.md
/okf-ingest raw/05_ad_mail.txt
~~~

💬 답 (Claude Code)

~~~text
[raw/02_mail_new_member.txt]
Creation: studio/people/eunchae에 새 문서를 만들었습니다(상태: draft). check.py에서 이번 작업으로 생긴 문제는 없고, 이미 있던 1건(playbooks/server_incident의 stale_after 2026-10-01 만료)만 남아 있습니다. 이번 자료와 상관없는 문서라 고치지 않았습니다.

- **새 문서 내용:** 서은채는 사운드 디자이너로 배경음과 효과음을 맡고, 11월 3일에 합류합니다. 첫 일은 초롱 원정대 출시 트레일러 음악이고, 자리는 한별 옆, 첫 주 온보딩은 오지호가 돕습니다. 각 사실 뒤에 메일 원문 각주를 달았고, 출시 프로젝트·한별·오지호 문서로 링크를 걸었습니다.
- **목차와 기록:** people/index.md에 한 줄을 더했고, log.md의 2026-11-02 아래에 Creation 기록을 남겼습니다.

메일에 연도가 없어서 합류일은 "11월 3일"로만 적었습니다.

[raw/03_data_memo_arppu.md]
Creation: game_data/metrics/arppu 문서를 새로 만들었고, 상태는 draft(초안)입니다.

- 위키에 ARPPU 문서가 없어서 지표 폴더에 새로 만들었습니다. 정의와 계산 방법을 적고, 결제 테이블 문서로 링크했습니다. 대시보드 계산은 아직 만들어지지 않아 정의만 둔다는 점도 적었습니다.
- 지표 목록과 데이터 위키 변경 기록에 이 문서를 한 줄씩 더했습니다.
- 검사 스크립트는 문제를 1개 보고했지만 이번 작업과 상관없습니다. 업무 위키의 서버 장애 절차 문서(studio/playbooks/server_incident)가 확인 기한(2026-10-01)을 넘겼다는 내용입니다. 이 문서는 고치라는 지시가 없어 그대로 두었습니다.

[raw/04_data_memo_conversion.md]
Update 완료: game_data/metrics/paying_conversion 문서를 deprecated로 바꿨습니다. 메모대로 이 지표는 더 쓰지 않으며, 팀마다 분모가 달라 숫자가 엇갈렸고 주간 보고와 대시보드에서 뺀다는 내용을 넣었습니다. log.md에도 한 줄 남겼습니다. check.py를 돌리니 문제 1개가 나왔는데, 이번에 고친 문서가 아니라 원래 있던 studio/playbooks/server_incident의 기한 지남(stale_after 2026-10-01)입니다.

[raw/05_ad_mail.txt]
건너뜀: 무선 이어폰 할인 쇼핑몰 광고 메일이라 위키(studio, game_data)에 남길 내용이 없습니다. 문서는 고치지 않았습니다.
~~~

**결과 읽기**
- **새 문서:** 사람 문서와 지표 문서는 `status: draft`로 만들어지고, 각 폴더 `index.md`에 한 줄씩 더해집니다.
- **deprecated:** 결제 전환율 문서는 상태가 `deprecated`로 바뀝니다. 에이전트가 스킬의 frontmatter 규칙('대상이 더는 현재가 아니면 deprecated')을 읽고 정합니다.
- **건너뜀:** 광고 메일은 위키에 아무것도 남기지 않습니다.

### 3-3. 어느 번들에도 맞지 않는 자료

`raw/06_lore_memo.md`는 게임 세계관(주인공, 무대, 악역) 메모입니다. 업무 위키(사람, 프로젝트, 회의, 업무 절차, 결정)에도 데이터 위키(데이터셋, 테이블, 지표)에도 맞지 않습니다. 넣기 스킬의 '새 번들 만들기' 규칙대로 에이전트가 새 번들을 만듭니다.

⌨️ 입력 (Claude Code)

~~~text
/okf-ingest raw/06_lore_memo.md
~~~

💬 답 (Claude Code)

~~~text
Creation(새 번들): 세계관 메모는 업무 위키(studio)와 데이터 위키(game_data) 어디에도 맞지 않아 `lore` 번들을 새로 만들고 `lore/world/chorong_expedition` 문서로 넣었습니다(상태: draft). 위키 검사 결과 새 번들에서는 문제가 없었고, 원래 있던 `studio/playbooks/server_incident`의 기한 지남(stale_after 2026-10-01) 1건은 이번 작업과 상관없어 그대로 두었습니다.
~~~

이 부록을 만들 때 생긴 새 번들의 파일입니다(모두 새 파일이라 모든 줄이 `+`).

~~~diff
===== lore/index.md
+---
+okf_version: "0.2"
+---
+# 초롱 원정대 세계관 위키
+
+* [세계관](world/) - 게임 속 인물, 장소, 이야기 설정

===== lore/log.md
+# 초롱 원정대 세계관 위키 변경 기록
+
+## 2026-11-02
+* **Initialization**: 번들을 만들었습니다.
+* **Creation**: 06_lore_memo.md를 반영해 [초롱 원정대 세계관](/world/chorong_expedition.md) 문서를 만들었습니다.

===== lore/world/index.md
+# 세계관
+
+* [초롱 원정대 세계관](chorong_expedition.md) - 주인공 초롱이, 밤하늘 섬과 별등, 악역 먹구름 설정

===== lore/world/chorong_expedition.md
+---
+type: Lore
+title: 초롱 원정대 세계관
+description: 주인공 초롱이, 밤하늘 섬과 별등, 악역 먹구름 설정.
+tags: [lore, character, world]
+status: draft
+sources:
+  - id: 06_lore_memo
+    resource: "raw://06_lore_memo.md"
+    title: 06_lore_memo.md
+generated: { by: "claude-code", at: "2026-11-02T09:00:00+09:00" }
+---
+# 주인공
+
+* 초롱이: 반딧불 마을에서 태어난 등불 요정[^06_lore_memo]
+* 목표: 일곱 별등을 다시 밝혀 밤하늘을 되찾는 것[^06_lore_memo]
+
+# 무대
+
+* 밤하늘 섬: 모두 일곱 개[^06_lore_memo]
+* 별등: 섬마다 꺼진 별등이 하나씩 있음[^06_lore_memo]
+
+# 악역
+
+* 먹구름: 빛을 먹는 그림자 고래[^06_lore_memo]
+
+[^06_lore_memo]: 06_lore_memo.md
~~~

**결과 읽기**
- **새 번들:** 맨 위 목차(`okf_version`), 변경 기록(Initialization, Creation), 폴더와 폴더 목차, 문서까지 OKF 번들의 모양을 갖춰 만들어집니다. 검사 스크립트와 부록 02의 비서는 `index.md`가 있는 폴더를 모두 번들로 보므로, 새 번들도 코드를 바꾸지 않고 함께 다룹니다.
- **사람 확인:** 새 번들은 위키의 구조를 바꾸는 일이라, 답에 "새 번들"이라고 밝힙니다. 이름과 폴더가 맞는지는 사람이 확인합니다.

### 3-4. 위키 점검

점검은 문서를 고치지 않습니다.

⌨️ 입력 (Claude Code)

~~~text
/okf-lint
~~~

⌨️ 입력 (Codex)

~~~text
위키를 점검해 줘
~~~

💬 답 (Claude Code, 3-1과 3-2의 다섯 건을 넣은 뒤)

~~~text
## OKF 위키 점검 결과 (2026-11-02)

check.py가 문제 1건을 찾았고, 오늘 바뀐 문서를 대조해 보니 출시일이 서로 다르게 적힌 곳이 1건 있습니다. 문서는 고치지 않았습니다.

### 1. check.py 결과

| 번들 | 점검 | 문서 | 내용 |
|---|---|---|---|
| studio | 만료 | playbooks/server_incident.md | stale_after가 2026-10-01이라 오늘(11-02) 기준으로 이미 지났습니다. 내용이 낡았을 수 있습니다. |

적합성, 끊긴 링크, 없는 원문 항목에서는 문제가 나오지 않았습니다.

### 2. 모순

| 문서 A의 줄 | 문서 B의 줄 |
|---|---|
| studio/projects/chorong_launch.md:21 `정식 출시: 12월 4일` (근거: 01_meeting_2026-10-31, 오늘 고침) | studio/playbooks/launch_checklist.md:12 `11월 20일 정식 출시 전에 아래 항목을 모두 끝냅니다.` (10-28 작성, 지호가 확인함) |

### 대조한 범위
- **오늘 바뀐 문서 A (log.md `## 2026-11-02` 기준):**
  - studio: projects/chorong_launch, people/eunchae
  - game_data: metrics/paying_conversion, metrics/arppu
- **함께 읽은 문서 B (A가 링크하거나 A를 링크하는 문서):**
  - studio: playbooks/launch_checklist, people/yuna, people/seoyeon, people/hanbyeol, people/jiho
  - game_data: tables/players, tables/purchases
- **문서 B에서 뺀 것:** meetings/2026-10-27_weekly는 회의 정리 문서라서 대조하지 않았습니다.
- **나머지 쌍은 서로 맞습니다.**
  - 박서연의 결제 일이 체크리스트 8번 "진행 중"과 맞습니다.
  - 한별의 자리, 오지호의 온보딩이 각 인물 문서와 맞습니다.
  - ARPPU의 "테스트 계정 제외"가 players 표의 `is_test` 열과 맞습니다.

출시일을 12월 4일과 11월 20일 가운데 어느 쪽으로 맞출지, server_incident를 다시 볼지는 정해 주시면 그에 따라 고치겠습니다.
~~~

**결과 읽기**
- **검사 결과:** `check.py`가 찾은 문제(만료 등)를 그대로 표로 옮깁니다.
- **모순:** 오늘 바뀐 프로젝트 문서의 출시일과, 바뀌지 않은 출시 체크리스트의 출시일이 다르다고 알립니다. 어느 쪽을 고칠지는 사람이 정하고, 고치라고 지시하면 에이전트가 넣기 스킬로 고칩니다.
- **따라 할 때 다를 수 있는 점:** 3-1을 Codex로 했다면 체크리스트도 이미 고쳐져 모순이 나오지 않을 수 있습니다. 3-3까지 했다면 새 번들의 문서도 오늘 바뀐 문서로 대조 범위에 나옵니다.

## 4. 핵심 정리

- **스킬 구성:** 스킬 하나는 `SKILL.md`(frontmatter와 본문)와 스크립트를 담은 폴더입니다. 저장 규칙은 본문에 적고, 형식은 검사 스크립트(`check.py`)로 확인합니다.
- **스킬 등록:** Claude Code는 `.claude/skills/`, Codex는 `.agents/skills/`에서 스킬을 찾으므로 같은 스킬을 두 폴더에 둡니다. 파일을 고칠 때 Claude Code는 스킬의 `allowed-tools`에 적은 도구를 묻지 않고 쓰고, Codex는 처음 실행할 때 폴더 사용을 허용받아 작업 폴더 안 파일을 고칩니다.
- **스킬 활용:** 지시가 스킬 설명과 맞으면 에이전트가 스스로 고르고, `/이름`(Claude Code), `$이름`(Codex)으로 직접 부를 수도 있습니다. 에이전트가 직접 쓰면 규칙을 어기는 줄이 생길 수 있어, 바뀐 줄은 사람이 확인합니다.
- **일반화:** 넣을 번들은 각 번들의 맨 위 목차를 읽고 고르고, 맞는 번들이 없으면 새 번들을 만듭니다. 그래서 스킬은 위키의 번들과 문서가 바뀌어도 그대로 씁니다.